# Day 4a. Reaching real systems

Every tool you have written this week was a mock dictionary. Two doors left:

- **MCP**, a standard plug, so a connector you build once works in *any*
  harness (your agent, Claude, the next framework)
- **Subagents**, a specialist with its own clean context, hidden behind one
  tool

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below, e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

## 1 · A connector is a tiny server

Same thinking as `@tool`: name, docstring, typed arguments. Only the transport
changed, it now lives outside your codebase, and anything that speaks MCP can
call it.

In [ ]:
%%writefile campus_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("campus")

TIMETABLE = {
    "Main": {"Mon": {"free": ["B12", "C01"], "booked": ["A04"]},
             "Tue": {"free": ["A04"],        "booked": ["B12", "C01"]}},
    "Annex": {"Mon": {"free": ["X01"],       "booked": []},
              "Tue": {"free": [],            "booked": ["X01"]}},
}

@mcp.tool()
def room_schedule(building: str, day: str) -> dict:
    """Free and booked rooms for a building and weekday."""
    return TIMETABLE.get(building, {}).get(day, {"error": "unknown building/day"})

@mcp.tool()
def list_buildings() -> list:
    """All buildings with a timetable."""
    return list(TIMETABLE)

if __name__ == "__main__":
    mcp.run()

That file is now a standalone server. Your notebook connects to it as a
**client**, and the tools arrive from outside.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.messages import HumanMessage

client = MultiServerMCPClient({
    "campus": {"command": "python", "args": ["campus_server.py"],
               "transport": "stdio"}})

tools = await client.get_tools()          # top-level await works in Jupyter
print("tools from the server:", [t.name for t in tools])

campus = create_agent(model=llm, tools=tools,
    system_prompt="Answer campus room questions using the tools. "
                  "Always name the building and day in your answer.")

out = await campus.ainvoke({"messages": [HumanMessage(
    "Is B12 free on Monday in the Main building?")]})
print("\n", out["messages"][-1].content)

Look at the agent code: **identical** to day 2. The agent cannot tell an MCP
tool from a local `@tool`.

What changed is *organizational*: the tool inventory is now config, which
servers, which tools, for which users. A file your security team can read.

## 2 · A subagent is an agent behind a tool

The clean-context idea, in eight lines: the researcher's twenty search results
burn **its** window. Only the conclusion comes back to the main agent.

In [ ]:
from langchain.tools import tool

SOURCES = {
    "mcp": "By 2026 every major assistant vendor ships MCP support; enterprises "
           "standardize their tool inventories on it to avoid per-vendor rewrites.",
    "evals": "Teams that run eval suites on real historical cases report far fewer "
             "production incidents than teams that test by trying things once.",
    "skills": "Written procedures raise pass rates sharply but cost more tokens "
              "and wall-clock time per run.",
}

@tool
def web_search(query: str) -> str:
    """Search the web (mock). Returns summaries of the top results."""
    hits = [v for k, v in SOURCES.items() if k in query.lower()]
    return " | ".join(hits) or "no results"

researcher = create_agent(model=llm, tools=[web_search],
    system_prompt="Research the question. Return dense findings, no chit-chat.")

@tool
def research(question: str) -> str:
    """Delegate a research question to a focused subagent."""
    out = researcher.invoke({"messages": [HumanMessage(question)]})
    return out["messages"][-1].content

briefer = create_agent(model=llm, tools=[research],
    system_prompt="You write one-paragraph briefings. Delegate research; never guess.")

out = briefer.invoke({"messages": [HumanMessage(
    "Brief me: why do teams standardize on MCP, and why do evals matter?")]})

print(out["messages"][-1].content)
print("\nmain agent's transcript is only", len(out["messages"]),
      "messages, the searching happened elsewhere")

> **Only when context demands it.** One agent until the window or the
> specialization forces more, never for style points. You will see this rule
> hold in all three cases in today's deck.

## 3 · Your turn, a guarded MCP write

Everything from this week, in one exercise:

1. Add a `book_room_slot(building, day, room)` tool to `campus_server.py` that
   returns a fake confirmation (edit the cell above and re-run it).
2. Restart the client so the new tool is picked up.
3. Guard it with **yesterday's** `HumanInTheLoopMiddleware`, plus a
   `checkpointer`, because pausing needs persistence.
4. Ask the agent to book a room, inspect the interrupt, approve it.

A tool that came from an external server, stopped by your middleware, resumed by
a human. That is a production pattern, in four cells.

In [ ]:
# your guarded MCP write here


---
## Wrap

MCP: the same tool thinking with a standard plug, build once, use in any
harness · subagents: specialists behind a `@tool`, protecting the main context ·
and both obey the guardrails you already know.

**Next (4b):** a complete system, and the evals that prove it works.